# orangejuice: forecasting (batch, automl-pipeline)

**Use case:** `usecase-type/batch/timeseries-forecasting/azure-automl/azureml-pipeline-with-automl-step` — bounded scoring of many requests into stored predictions with the Azure ML pipeline with an AutoML step.

**Training:** ESML pipeline factory `IN_2_GOLD_TRAINING_AUTOML` (medallion lake, data assets, quality-gated evaluation) driven by AzureMLRollout receipts.

**Serving:** ESML GOLD_INFERENCE plan published as a pipeline-component batch endpoint, or submitted as a pipeline job, writing predictions to the lake.

**Status:** Executable notebook templates over shared engines; cloud steps are explicit, charged operations behind switches that default to false and need the listed prerequisites.

Kaggle-only source: `NOT SELECTED`.
Dataset status: **required-selection**.
BLOCKED: select and verify a Kaggle orange-juice sales time series, schema, version and license with the user. No suitable Kaggle source has been verified. mathsian/orange-juice is ISLR brand-purchase classification, NOT dated sales forecasting. Ask the user before substituting another source or use case; there is no automatic fallback. The following schema is an explicit selection contract, not a claim about an existing dataset.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- Install the factory extras `train` (plus `vision` for images, `azure` for Azure ML steps and `streaming` for Event Hubs consumers).
- For Azure steps: `user-config/runtime.local.json` naming an existing workspace, compute, identity and storage selection.
- `user-config/esml-rollout.local.json` with a reviewed source binding for this scenario and pinned Azure ML environments; the ESML SDK is imported from `esml-v2` or azure-esml-sdk.
- ESML AutoML forecast inference needs `inference_history_path` (observed history in the lake) in the scenario's reviewed rollout binding.
- AutoML models are scored in Azure with an AutoML-compatible environment, not in the local venv.

**Limitations**

- AutoML forecasting needs observed history with each scoring request (history.parquet or the request's history block).
- Pipeline-component batch invocation failed with an unresolved service URI error in the paused Dev rollout; submit the same inference plan as a pipeline job if invocation is unavailable.
- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "orangejuice.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "automl"
PATTERN = "batch"
TECHNOLOGY = "azure-automl/azureml-pipeline-with-automl-step"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "automl-pipeline"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "automl-pipeline" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RUN_INGEST = False
RUN_PREPARE = False
PREPARE_ESML = False
RUN_ESML_TRAINING = False
SAMPLE_REQUESTS = False
PREPARE_ESML_INFERENCE = False
PUBLISH_AND_INVOKE_ESML = False
SUBMIT_ESML_INFERENCE = False
RUN_ID = None  # defaults to the reviewed binding's run_id; use a new id per attempt
COMPONENT_VERSION = ""  # new pinned pipeline component version, e.g. "1"
SELECTION_POLICY = None  # e.g. ROOT / "user-config" / "model" / "model-selection.json"
NO_CHAMPION = False
INFERENCE_RUN_ID = ""
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Kaggle ingestion and reproducible preparation

Review the source license and any competition rules first. Required-selection and
license-review gates block ingestion. Do not clear them without documented approval.
The download manifest records file hashes and pinned dataset version. Existing nonempty
outputs are never silently replaced. Titanic preparation additionally writes `titanic.parquet`.
Forecasting uses chronological holdouts, not random rows.


In [ ]:
if RUN_INGEST:
    factory("ingest", "--scenario", SCENARIO, "--output", RAW)
INPUT = RAW / scenario["dataset"]["file"]
if RUN_PREPARE:
    factory("prepare", "--scenario", SCENARIO, "--input", INPUT, "--output", PREPARED)
if (PREPARED / "manifest.json").exists():
    print(json.dumps(load_json(PREPARED / "manifest.json"), indent=2))


## ESML pipeline factory: `IN_2_GOLD_TRAINING_AUTOML`

The reusable ESML v2 pipeline factory compiles this scenario's reviewed source binding from
`user-config/esml-rollout.local.json` (schema `esml.azureml-rollout-config/v1`, see the
example file) into a medallion IN-to-BRONZE-to-SILVER-to-GOLD pipeline with a leakage-aware
split, `IN_2_GOLD_TRAINING_AUTOML` and held-out evaluation. `AzureMLRollout` records every cloud write as a
receipt; it never resubmits after an uncertain outcome. Preparation is offline.


In [ ]:
ESML_CONFIG = ROOT / "user-config" / "esml-rollout.local.json"
ESML_OUTPUT = WORK / "esml"


def esml_rollout(cloud=False):
    """Import the ESML SDK (installed or from the purple/orange checkout) and bind this scenario."""
    try:
        import azure_esml  # noqa: F401
    except ModuleNotFoundError:
        for candidate in (ROOT.parents[1] / "esml-v2", ROOT.parents[1] / "azure-enterprise-scale-ml" / "esml-v2"):
            if (candidate / "azure_esml" / "__init__.py").is_file():
                sys.path.insert(0, str(candidate))
                break
        else:
            raise RuntimeError("Install azure-esml-sdk or open this notebook inside the purple/orange checkout")
    from azure_esml import ESMLProject
    from azure_esml.domain_layer.rollout import AzureMLRollout, project_from_rollout_config
    config = load_json(ESML_CONFIG)
    project = project_from_rollout_config(config, ROOT / "user-config" / "model" / "scenarios",
                                          scenario=scenario["name"], mode=MODE, base_path=ESML_CONFIG.parent)
    if cloud:
        from azure_esml.base_layer import AzureMLSDKBackend
        project = ESMLProject(project.settings, backend=AzureMLSDKBackend.from_cli(project.target, subscription_bound=True))
    return AzureMLRollout(project, ESML_OUTPUT), config


if PREPARE_ESML or RUN_ESML_TRAINING:
    rollout, config = esml_rollout(cloud=RUN_ESML_TRAINING)
    binding = config["scenarios"][scenario["name"]]
    RUN_ID = RUN_ID or binding["run_id"]
    from azure_esml.domain_layer.rollout import SCENARIO_NUMBERS
    if PREPARE_ESML:
        request = rollout.prepare(model_number=SCENARIO_NUMBERS[scenario["name"]], mode=MODE,
                                  data_date_utc=config["data_date_utc"], run_id=RUN_ID,
                                  data_version=binding["data_version"])
        print(json.dumps(request, indent=2))
    if RUN_ESML_TRAINING:
        if not COMPONENT_VERSION:
            raise ValueError("Set COMPONENT_VERSION to a new pinned pipeline component version")
        if not (ESML_OUTPUT / RUN_ID / "registered-definitions.json").exists():
            rollout.register_definitions(RUN_ID, component_version=COMPONENT_VERSION)
        policy = load_json(SELECTION_POLICY) if SELECTION_POLICY else None
        print(json.dumps(rollout.execute_training(RUN_ID, selection_policy=policy, no_champion=NO_CHAMPION), indent=2))
if RUN_ID and (ESML_OUTPUT / RUN_ID / "training-completion.json").exists():
    MODEL_ID = load_json(ESML_OUTPUT / RUN_ID / "training-completion.json")["model_id"]
    MODEL_NAME = load_json(ESML_OUTPUT / RUN_ID / "request.json")["model_name"]
    print("MODEL_ID:", MODEL_ID)


## Unlabeled requests from the held-out split

`sample-requests` writes demonstration requests without labels; `requests.parquet` is the unlabeled batch input.
`labels.parquet` keeps held-out truth separately for monitoring demos, and forecasting
also writes `history.parquet` for AutoML scoring. Never send labels to a model.


In [ ]:
if SAMPLE_REQUESTS:
    factory("sample-requests", "--scenario", SCENARIO, "--prepared", PREPARED, "--output", REQUESTS, "--rows", 10)
if (REQUESTS / "manifest.json").exists():
    print(json.dumps(load_json(REQUESTS / "manifest.json"), indent=2))


## ESML batch inference (`GOLD_INFERENCE`)

`prepare_inference` renders an inference plan bound to the registered model version and a
new inference run ID; DataOps must place unlabeled gold features with `request_id` at the
printed input path. Publish and invoke a pipeline-component batch endpoint (alternative A)
or submit the same plan as a pipeline job (alternative B). Receipts prevent repeats.


In [ ]:
if PREPARE_ESML_INFERENCE or PUBLISH_AND_INVOKE_ESML or SUBMIT_ESML_INFERENCE:
    if PUBLISH_AND_INVOKE_ESML and SUBMIT_ESML_INFERENCE:
        raise ValueError("Choose one inference route to avoid duplicate charged jobs")
    rollout, config = esml_rollout(cloud=PUBLISH_AND_INVOKE_ESML or SUBMIT_ESML_INFERENCE)
    RUN_ID = RUN_ID or config["scenarios"][scenario["name"]]["run_id"]
    if PREPARE_ESML_INFERENCE:
        if not INFERENCE_RUN_ID:
            raise ValueError("Set INFERENCE_RUN_ID to a new inference run identifier")
        print(json.dumps(rollout.prepare_inference(RUN_ID, inference_date_utc=config["data_date_utc"],
                                                   inference_run_id=INFERENCE_RUN_ID), indent=2))
    if PUBLISH_AND_INVOKE_ESML:
        if not COMPONENT_VERSION:
            raise ValueError("Set COMPONENT_VERSION to a new pinned inference component version")
        print(json.dumps(rollout.deploy_and_invoke(RUN_ID, component_version=COMPONENT_VERSION), indent=2))
    if SUBMIT_ESML_INFERENCE:
        print(json.dumps(rollout.submit_inference(RUN_ID), indent=2, default=str))


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Batch compute scales back to zero; storage of inputs and predictions remains.
